# 00 · Setup and script kit
**CVES manuscript repair series.** This notebook prepares everything the other five need. Run it once, then again only if you start over.

## The series
| Notebook | Guide steps | What it does |
|---|---|---|
| **00 Setup** (this one) | 0 | Unpacks the supervisor's script kit, the reference reports, and the dataset |
| **01 Clock and acquisition** | 0, 1, 2 | Known-answer tests, corrected 1000 Hz clock, plausibility gate, clock figure |
| **02 Clinical** | 7, 8 | Clinical sets R/P/A, n = 64 and n = 91, Shapley blocks, coefficients, leak checks |
| **03 Phase grid** | 3, 4 | Full phase grid, the draft-clock exhibit, full curves (Figure S1) |
| **04 Tests and robustness** | 2, 4, 5, 6 | Regime definition, enrolment order, bias, repeatability, increment test, shortcut test |
| **05 Sister datasets, ledger, freeze** | 10, 12, 15, 16 | Clock check of same-lab datasets, number ledger, manifest and package |

## Before you start (once)
1. Save the supervisor's markdown file as **`guide.md`** into your Google Drive folder `MyDrive/CVES Fresh/CVES Repair/` (the folder already exists; or upload when asked).
2. Have the CVES v1.0.0 dataset ready in Drive (folder or zip), **or** let the notebook download it from PhysioNet. The kit does **not** need the 77 GB `raw-data.tar.gz`.
3. Use a standard Colab runtime (CPU is enough). Expect 15–30 minutes in total for notebooks 01–04.

## Paste map: which report feeds which manuscript item
| Manuscript item (guide Part 4) | Report |
|---|---|
| A1, S1, R1 (clock, heart rate, draft clock) | `1_acquisition`, `6_draft_clock` |
| S2, R2 (regime, scale, enrolment order) | `7_robustness` |
| R3, Table 5, Figure 4 (phase grid, bias) | `4_grid`, `7_robustness` |
| R4 (repeatability) | `7_robustness`, `5_tests` |
| R5 (signal increment) | `5_tests` |
| R6 (shortcut test) | `7_robustness` |
| R7, S4 (clinical, Shapley, coefficients) | `3_clinical`, `7_robustness` |
| Figure S1 (full curves) | `8_curves` → `figS1_curves.csv` |

In [ ]:
#@title 1. Mount Drive and create folders { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/CVES Fresh/CVES Repair')
for d in ['kit', 'private', 'reports', 'reference', 'figures']:
    (BASE / d).mkdir(parents=True, exist_ok=True)
print('Folders ready under', BASE)
print('private/  = per-participant files. Never share this folder.')
print('reports/  = aggregate text reports. Safe to share with your supervisor.')

In [ ]:
HELPERS_SRC = r'''# helpers.py - shared by every notebook in the CVES repair series (written by notebook 00).
# Plain: runs the supervisor's script kit step by step, compares each report with the supervisor's reference run,
# parses the reports into tables, and draws the figures. It never reads group labels itself.
import os, re, ast, sys, json, shutil, hashlib, platform, subprocess, time, zipfile
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd

BASE = Path(os.environ.get("CVES_BASE", "/content/drive/MyDrive/CVES Fresh/CVES Repair"))
CFG_FILE = BASE / "config.json"
NUM = re.compile(r"[-+−]?\d+(?:\.\d+)?(?:e[-+]?\d+)?", re.I)
REPORTS = ["0_known_answers", "1_acquisition", "3_clinical", "4_log", "4_grid", "5_tests", "6_draft_clock", "7_robustness", "8_curves"]


# ---------------------------------------------------------------- paths and config
def paths():
    return SimpleNamespace(base=BASE, kit=BASE / "kit", work=BASE / "private", reports=BASE / "reports",
                           ref=BASE / "reference", figs=BASE / "figures", guide=BASE / "guide.md")


def save_cfg(**kw):
    BASE.mkdir(parents=True, exist_ok=True)
    CFG_FILE.write_text(json.dumps(kw, indent=2))


def setup(need_data=True):
    """Load config, put the kit on sys.path, make sure the dataset is present on the local disk."""
    p = paths()
    if not CFG_FILE.exists():
        raise SystemExit("Run notebook 00_Setup first (config.json is missing).")
    c = json.loads(CFG_FILE.read_text())
    for d in (p.work, p.reports, p.figs, p.ref):
        d.mkdir(parents=True, exist_ok=True)
    if str(p.kit) not in sys.path:
        sys.path.insert(0, str(p.kit))
    os.chdir(p.kit)
    cfg = SimpleNamespace(**vars(p), data=Path(c["data"]), drive_data=c.get("drive_data", ""), download=c.get("download", False))
    if need_data:
        ensure_data(cfg)
    print("Kit:", cfg.kit, "\nData:", cfg.data, "\nPrivate (never share):", cfg.work, "\nReports (aggregates only):", cfg.reports)
    return cfg


def ensure_data(cfg):
    if (cfg.data / "subjects.csv").exists():
        return
    print("Dataset is not on this runtime's disk yet. Restoring it ...")
    if cfg.drive_data:
        get_from_drive(cfg.drive_data, cfg.data)
    elif cfg.download:
        download_physionet(cfg.data)
    if not (cfg.data / "subjects.csv").exists():
        raise SystemExit("No subjects.csv under %s. Re-run notebook 00_Setup, section 4." % cfg.data)


def get_from_drive(src, dst):
    src = Path(src)
    dst = Path(dst)
    if src.suffix.lower() == ".zip":
        with zipfile.ZipFile(src) as z:
            z.extractall(dst.parent / "_unzip")
        root = next(iter(sorted((dst.parent / "_unzip").rglob("subjects.csv"))), None)
        if root is None:
            raise SystemExit("No subjects.csv inside the zip.")
        shutil.copytree(root.parent, dst, dirs_exist_ok=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("raw-data.tar.gz", "*.tar.gz"))


def download_physionet(dst):
    # Skips the 77 GB raw archive; the kit does not need it.
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    cmd = ["wget", "-q", "-r", "-N", "-c", "-np", "-nH", "--cut-dirs=3", "-e", "robots=off", "-R", "*.tar.gz",
           "-P", str(dst), "https://physionet.org/files/cves/1.0.0/"]
    print("Downloading (this can take a while):", " ".join(cmd))
    subprocess.run(cmd, check=False)


def check_data(cfg, expect={"head-up-tilt": 68, "sit-stand": 71, "sit-stand-balance": 64}):
    ok = (cfg.data / "subjects.csv").exists()
    print("subjects.csv:", "found" if ok else "MISSING")
    for proto, n in expect.items():
        k = len(list((cfg.data / "data" / proto).glob("*.hea")))
        print("  %-18s %3d records (the supervisor's run had %d)  %s" % (proto, k, n, "OK" if k == n else "CHECK"))
        ok &= (k == n)
    return ok


def check_private_folder(cfg):
    r = subprocess.run(["git", "-C", str(cfg.work), "rev-parse"], capture_output=True, text=True)
    if r.returncode == 0:
        raise SystemExit("Stop: the private folder is inside a git repository. Choose another folder.")
    print("Private folder is not inside a git repository: OK")


# ---------------------------------------------------------------- the kit and the reference reports
def unpack_kit(guide, kit_dir):
    t = Path(guide).read_text(encoding="utf-8")
    Path(kit_dir).mkdir(parents=True, exist_ok=True)
    found = re.findall(r"<!-- kit-file: (\S+) -->\n~~~~[a-z]*\n(.*?)\n~~~~\n", t, re.S)
    for name, body in found:
        (Path(kit_dir) / name).write_text(body + "\n", encoding="utf-8")
    return [n for n, _ in found]


def compile_kit(kit_dir):
    bad = []
    for f in sorted(Path(kit_dir).glob("*.py")):
        r = subprocess.run([sys.executable, "-m", "py_compile", str(f)], capture_output=True, text=True)
        if r.returncode:
            bad.append((f.name, r.stderr.strip().splitlines()[-1]))
    return bad


def extract_reference(guide, ref_dir):
    t = Path(guide).read_text(encoding="utf-8")
    Path(ref_dir).mkdir(parents=True, exist_ok=True)
    out = {}
    for title, body in re.findall(r"^## ([^\n]+)\n\n(?:[^~\n][^\n]*\n\n)?~~~~text\n(.*?)\n~~~~", t, re.S | re.M):
        name = title.split(":")[0].strip()
        if name.startswith("Raw-archive"):
            name = "raw_census"
        elif name.startswith("Same-laboratory"):
            name = "external_headers"
        if name[:1].isdigit() or name in ("raw_census", "external_headers"):
            (Path(ref_dir) / (name + ".txt")).write_text(body + "\n", encoding="utf-8")
            out[name] = len(body.splitlines())
    return out


# ---------------------------------------------------------------- running a kit step
def run_step(cfg, name, script, *args):
    """Run one kit script, stream its output, save it as reports/<name>.txt. Stops at the first failure, like run_all.sh."""
    cmd = [sys.executable, "-u", script] + [str(a) for a in args]
    print("=== %s: %s" % (name, " ".join(cmd[2:])), flush=True)
    t0, lines = time.time(), []
    p = subprocess.Popen(cmd, cwd=str(cfg.kit), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for ln in p.stdout:
        lines.append(ln)
        print(ln, end="")
    rc = p.wait()
    txt = "".join(lines)
    (cfg.reports / (name + ".txt")).write_text(txt, encoding="utf-8")
    print("--- %s finished in %.0f s (exit code %d); saved to %s" % (name, time.time() - t0, rc, cfg.reports / (name + ".txt")))
    if rc != 0:
        raise RuntimeError("Step %s failed. Read the last lines above; do not patch individual records." % name)
    return txt


def read_report(cfg, name):
    return (cfg.reports / (name + ".txt")).read_text(encoding="utf-8")


# ---------------------------------------------------------------- comparing with the supervisor's reference run
def _skel(line):
    return NUM.sub("#", line).strip()


def _tok(line):
    return [t.replace("−", "-") for t in NUM.findall(line)]


def _close(a, b):
    if "." not in a and "." not in b and "e" not in a.lower() and "e" not in b.lower():
        return float(a) == float(b)
    x, y = float(a), float(b)
    tol = 0.011 if max(abs(x), abs(y)) < 1.5 else 1.0
    return abs(x - y) <= tol


def compare(cfg, name, show_ok=False):
    """Line-by-line comparison. Rule of thumb from the guide: counts exact, AUC +/-0.01, phase and timing +/-1."""
    rf = cfg.ref / (name + ".txt")
    if not rf.exists():
        print("No reference report for", name)
        return None
    ref = [l for l in rf.read_text(encoding="utf-8").splitlines() if l.strip()]
    mine = [l for l in read_report(cfg, name).splitlines() if l.strip()]
    pool = {}
    for l in mine:
        pool.setdefault(_skel(l), []).append(l)
    rows, ok, missing = [], 0, 0
    for l in ref:
        if not NUM.search(l):
            continue
        cand = pool.get(_skel(l))
        if not cand:
            missing += 1
            rows.append(dict(status="line differs or is missing", reference=l.strip()[:150], yours=""))
            continue
        m = cand.pop(0)
        a, b = _tok(l), _tok(m)
        bad = [(x, y) for x, y in zip(a, b) if not _close(x, y)]
        if bad:
            rows.append(dict(status="numbers differ", reference=l.strip()[:150], yours=m.strip()[:150]))
        else:
            ok += 1
            if show_ok:
                rows.append(dict(status="ok", reference=l.strip()[:150], yours=m.strip()[:150]))
    df = pd.DataFrame(rows)
    n_all = ok + len(df[df.status != "ok"]) if len(df) else ok
    print("%s: %d of %d numeric reference lines match within tolerance; %d differ or are missing." % (name, ok, n_all, n_all - ok))
    if len(df):
        pd.set_option("display.max_colwidth", 160, "display.width", 250)
        try:
            from IPython.display import display
            display(df[df.status != "ok"] if not show_ok else df)
        except Exception:
            print(df)
    else:
        print("All matching. Differences of a few degrees are expected only where the guide says so (library versions, implementation).")
    return df


# ---------------------------------------------------------------- parsing the reports
def parse_np(s):
    return ast.literal_eval(re.sub(r"np\.float64\(([^)]*)\)", r"\1", s))


GRID = re.compile(r"^\s*(\S+)\s+(\S+)\s+([LR])\s+(CARNet|surrogate|all bins)\s+(un|pr|ro)\s+(\d+)\s+(\d+)\s+(\d+)" + r"\s+([-+]?\d+\.\d+)" * 7 + r"\s*$")


def parse_grid(text):
    rows = []
    for ln in text.splitlines():
        m = GRID.match(ln)
        if m:
            g = m.groups()
            rows.append(dict(proto=g[0], seg=g[1], side=g[2], gate=g[3], arm=g[4], n=int(g[5]), S=int(g[6]), C=int(g[7]),
                             b=float(g[8]), lo=float(g[9]), hi=float(g[10]), p=float(g[11]), perm=float(g[12]), rob=float(g[13]), lev=float(g[14])))
    d = pd.DataFrame(rows)
    if len(d):
        d["protocol"] = d.proto.map({"head": "head-up tilt", "sit-": "sit-stand-balance"}).fillna(d.proto)
        d["clock"] = np.where(d.seg == "HEADERCLOCK", "labelled 500 Hz", "corrected")
    return d


def clinical_tables(t3, t7):
    out = {}
    m = re.search(r"^point \(100 reps\): (\{.*\})$", t3, re.M)
    a = parse_np(m.group(1)) if m else {}
    m = re.search(r"^all completed visits n (\d+) stroke (\d+) gait missing \d+ (\{.*\})", t3, re.M)
    b = parse_np(m.group(3)) if m else {}
    out["auc"] = pd.DataFrame({"n = 64": pd.Series(a), "n = 91 (all completed visits)": pd.Series(b)}).loc[lambda x: x.index.isin(["R", "P", "A", "R+gait", "R+meds", "gait", "meds"])]
    sh = {}
    for m in re.finditer(r"^n=(\d+) stroke (\d+) (\{.*\})$", t7, re.M):
        for k, (v, ci) in parse_np(m.group(3)).items():
            sh.setdefault("n = " + m.group(1), {})[k] = "%.3f (%.3f to %.3f)" % (v, ci[0], ci[1])
    out["shapley"] = pd.DataFrame(sh)
    rows = []
    for m in re.finditer(r"^\s+(\w+)\s+beta ([-+]\d+\.\d+); 95% \[([-\d.]+), ([-\d.]+)\]; top-ranked (\d+)%; same sign (\d+)%", t3, re.M):
        rows.append(dict(variable=m.group(1), beta=float(m.group(2)), lo=float(m.group(3)), hi=float(m.group(4)), top_ranked_pct=int(m.group(5)), same_sign_pct=int(m.group(6))))
    out["coef"] = pd.DataFrame(rows)
    return out


# ---------------------------------------------------------------- figures
def _plt():
    import matplotlib
    matplotlib.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
    import matplotlib.pyplot as plt
    return plt


def clock_figure(step1_pkl, out):
    """Heart rate and breathing: sit-stand-balance as labelled (500 Hz), as corrected (1000 Hz), and the supine tilt session."""
    plt = _plt()
    d = pd.read_pickle(step1_pkl)
    s, t = d[d.proto == "sit-stand-balance"], d[d.proto == "head-up-tilt"]
    cols = [("Seated, as labelled\n(500 Hz)", s.hr / 2, s.br / 2), ("Seated, corrected\n(1000 Hz)", s.hr, s.br), ("Supine session\n(head-up tilt)", t.hr, t.br)]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    rng = np.random.default_rng(0)
    for k, (title, lo, hi, idx) in enumerate([("Heart rate (per minute)", 40, 110, 1), ("Breathing rate (per minute)", 6, 30, 2)]):
        a = ax[k]
        a.axhspan(lo, hi, color="#2a9d8f", alpha=.12, label="plausible range")
        for i, c in enumerate(cols):
            y = c[idx].dropna().values
            a.boxplot([y], positions=[i], widths=.5, showfliers=False)
            a.scatter(i + rng.uniform(-.18, .18, len(y)), y, s=9, color="#264653", alpha=.6)
        a.set_xticks(range(3))
        a.set_xticklabels([c[0] for c in cols])
        a.set_title(title)
        a.legend(loc="upper right", frameon=False)
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    return fig


def clock_gate(step1_pkl):
    d = pd.read_pickle(step1_pkl)
    x = d[d.proto == "sit-stand-balance"]
    ok = x.hr.between(40, 110) & x.br.between(6, 30)
    share = ok.mean()
    print("sit-stand-balance at 1000 Hz: %d of %d records pass the plausibility gate (%.0f%%). The guide requires at least 90%%." % (ok.sum(), len(x), 100 * share))
    return share >= .9


def grid_forest(g, out):
    plt = _plt()
    g = g.copy()
    g["label"] = g.protocol.str.replace("sit-stand-balance", "seated") + " " + g.seg + " | " + g.arm
    order = list(dict.fromkeys(g.label))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), sharey=True, sharex=True)
    col = {"un": "#264653", "pr": "#e76f51", "ro": "#2a9d8f"}
    for i, side in enumerate("LR"):
        for j, gate in enumerate(["CARNet", "surrogate", "all bins"]):
            a = axes[i, j]
            x = g[(g.side == side) & (g.gate == gate)]
            for _, r in x.iterrows():
                y = order.index(r.label)
                a.plot([r.lo, r.hi], [y, y], color=col[r.arm], lw=1.6, alpha=.5 if r.seg == "HEADERCLOCK" else 1, ls="--" if r.seg == "HEADERCLOCK" else "-")
                a.plot(r.b, y, "o", color=col[r.arm], ms=5)
            a.axvline(0, color="grey", lw=.8)
            a.set_title("%s channel, %s gate" % ("Left" if side == "L" else "Right", gate))
            a.set_yticks(range(len(order)))
            a.set_yticklabels(order, fontsize=7)
            a.set_xlabel("stroke minus control (degrees)")
    axes[0, 0].invert_yaxis()
    fig.suptitle("Phase grid: every clock x segment x gate x alignment x side. Dashed = labelled (wrong) clock. un = unaligned, pr = per-record, ro = regime-offset.", fontsize=10)
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    return fig


def curves_figure(csv, out):
    plt = _plt()
    c = pd.read_csv(csv)
    fig, ax = plt.subplots(3, 2, figsize=(11, 9), sharex=True)
    for j, proto in enumerate(["head-up-tilt", "sit-stand-balance"]):
        for i, (v, lab) in enumerate([("phase", "Phase (degrees)"), ("gain", "Gain (%/%)"), ("coh", "Coherence")]):
            for grp, x in c[c.proto == proto].groupby("group"):
                ax[i, j].plot(x.f, x[v], label=str(grp))
            ax[i, j].axvspan(.07, .20, color="grey", alpha=.12)
            ax[i, j].set_ylabel(lab)
        ax[0, j].set_title(proto)
        ax[2, j].set_xlabel("Frequency (Hz); grey = low-frequency band 0.07-0.20 Hz")
    ax[0, 0].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    return fig


def coef_figure(coef, out):
    plt = _plt()
    c = coef.sort_values("beta")
    fig, a = plt.subplots(figsize=(6, 5))
    a.errorbar(c.beta, range(len(c)), xerr=[c.beta - c.lo, c.hi - c.beta], fmt="o", color="#264653", ecolor="#8ab", capsize=2)
    a.axvline(0, color="grey", lw=.8)
    a.set_yticks(range(len(c)))
    a.set_yticklabels(c.variable)
    a.set_xlabel("standardised coefficient (bootstrap 95% interval)")
    fig.tight_layout()
    fig.savefig(out, dpi=200)
    return fig


# ---------------------------------------------------------------- freezing
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def manifest(cfg):
    import importlib.metadata as md
    pk = {}
    for n in ["numpy", "scipy", "pandas", "statsmodels", "scikit-learn", "wfdb", "matplotlib"]:
        try:
            pk[n] = md.version(n)
        except Exception:
            pk[n] = "not installed"
    m = dict(date=time.strftime("%Y-%m-%d %H:%M:%S"), python=platform.python_version(), packages=pk,
             kit={f.name: sha256(f) for f in sorted(cfg.kit.glob("*")) if f.is_file()},
             reports={f.name: sha256(f) for f in sorted(cfg.reports.glob("*.txt"))},
             figures={f.name: sha256(f) for f in sorted(cfg.figs.glob("*")) if f.is_file()})
    (cfg.reports / "MANIFEST.json").write_text(json.dumps(m, indent=2))
    return m


def package(cfg, out):
    """Zip the kit, reports and figures only. Per-participant files (pkl) and the dataset are never included."""
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for folder, pat in ((cfg.kit, "*"), (cfg.reports, "*"), (cfg.figs, "*")):
            for f in sorted(folder.glob(pat)):
                if f.is_file() and f.suffix not in (".pkl", ".pyc"):
                    z.write(f, folder.name + "/" + f.name)
    return out


# ---------------------------------------------------------------- small readers
def section(text, key):
    """Lines of the report block that starts with '== <key...>' up to the next '==' line."""
    out, on = [], False
    for ln in text.splitlines():
        if ln.startswith("=="):
            on = ln[2:].strip().lower().startswith(key.lower())
            if on:
                out.append(ln)
            continue
        if on:
            out.append(ln)
    return "\n".join(out)


def grep(text, pattern):
    return "\n".join(l for l in text.splitlines() if re.search(pattern, l))


DRAFT = re.compile(r"^(\S+) n (\d+) \((\d+)/(\d+)\) ([+-]\d+\.\d+) \((-?\d+\.\d+), (-?\d+\.\d+)\) p (\S+); median phase (\S+)", re.M)


def core_exhibit(t6):
    """The supervisor's core exhibit: same samples, labelled clock vs correct clock (from 6_draft_clock)."""
    name = {"H_LF": "Header clock (500 Hz, as the draft used), 0.07-0.20 Hz", "T_LF": "Correct clock (1000 Hz), true low-frequency band 0.07-0.20 Hz",
            "T_0.14-0.40": "Correct clock, the band the draft really measured (0.14-0.40 Hz)"}
    arm = {"un": "unaligned", "all": "unaligned, all bins", "al": "aligned"}
    rows = []
    for m in DRAFT.finditer(t6):
        k, a = m.group(1).rsplit("_", 1)
        rows.append(dict(analysis=name.get(k, k), gate_or_arm=arm.get(a, a), n="%s (%s/%s)" % m.group(2, 3, 4), stroke_minus_control_deg=float(m.group(5)),
                         ci=(float(m.group(6)), float(m.group(7))), p=m.group(8), median_phase_deg=float(m.group(9))))
    return pd.DataFrame(rows)


INCR = re.compile(r"^(\S+) (\w+) (A|R) n (\d+) S (\d+) base ([\d.]+) plus ([\d.]+) delta ([-+][\d.]+) CI \[([-\d.]+), ([-\d.]+)\]", re.M)


def increment_rule(t5):
    """RQ4 rule from the analysis lock: an increment counts only if the refit CI is above 0 and the gain exceeds 0.04."""
    rows = []
    for m in INCR.finditer(t5):
        d, lo, hi = float(m.group(8)), float(m.group(9)), float(m.group(10))
        rows.append(dict(protocol=m.group(1), alignment=m.group(2), set=m.group(3), n=int(m.group(4)), base=float(m.group(6)), plus=float(m.group(7)),
                         delta=d, ci_lo=lo, ci_hi=hi, counts_as_increment=bool(lo > 0 and d > 0.04)))
    return pd.DataFrame(rows)


def rq3_rule(g):
    """RQ3 rule: claim a difference only if both alignments (per-record and regime-offset) exclude 0 with the same sign under both gates."""
    g = g[g.clock == "corrected"]
    rows = []
    for (pr, seg, sd), x in g.groupby(["protocol", "seg", "side"]):
        res = {}
        for gate in ("CARNet", "all bins"):
            y = x[(x.gate == gate) & x.arm.isin(["pr", "ro"])]
            sign = [np.sign(r.b) if (r.hi < 0 or r.lo > 0) else 0 for _, r in y.iterrows()]
            res[gate] = len(sign) == 2 and sign[0] != 0 and sign[0] == sign[1]
        rows.append(dict(protocol=pr, segment=seg, side=sd, main_gate=res["CARNet"], all_bins=res["all bins"], difference_claimable=res["CARNet"] and res["all bins"]))
    return pd.DataFrame(rows)
'''

from pathlib import Path
Path('/content/drive/MyDrive/CVES Fresh/CVES Repair/kit').mkdir(parents=True, exist_ok=True)
Path('/content/drive/MyDrive/CVES Fresh/CVES Repair/kit/helpers.py').write_text(HELPERS_SRC, encoding='utf-8')
print('helpers.py written:', len(HELPERS_SRC.splitlines()), 'lines')

In [ ]:
#@title 2. Parameters { display-mode: "form" }
GUIDE_MD = "/content/drive/MyDrive/CVES Fresh/CVES Repair/guide.md" #@param {type:"string"}
DATASET_IN_DRIVE = "/content/drive/MyDrive/Stroke In CVES/data/cerebral-vasoregulation" #@param {type:"string"}
DOWNLOAD_FROM_PHYSIONET = False #@param {type:"boolean"}
# DATASET_IN_DRIVE: path to the CVES v1.0.0 folder or a .zip in your Drive (leave blank if downloading).
# DOWNLOAD_FROM_PHYSIONET: tick only if the dataset is not in Drive. The raw 77 GB archive is skipped.

In [ ]:
#@title 3. Install packages, unpack the kit, extract the reference reports { display-mode: "form" }
!pip -q install wfdb
import sys, shutil; sys.path.insert(0, '/content/drive/MyDrive/CVES Fresh/CVES Repair/kit')
import helpers as H
from pathlib import Path
p = H.paths()
if Path(GUIDE_MD).exists() and Path(GUIDE_MD) != p.guide:
    shutil.copy(GUIDE_MD, p.guide)
if not p.guide.exists():
    from google.colab import files
    print('guide.md not found. Upload the supervisor markdown file now.')
    up = files.upload(); name = next(iter(up)); p.guide.write_bytes(up[name])
names = H.unpack_kit(p.guide, p.kit)
print('Kit files written:', ', '.join(names))
bad = H.compile_kit(p.kit)
print('Compile check:', 'all kit scripts compile' if not bad else bad)
print('Reference reports extracted (lines):', H.extract_reference(p.guide, p.ref))

In [ ]:
#@title 4. Copy the dataset to this runtime { display-mode: "form" }
import re, shutil, importlib, sys
from pathlib import Path

# 1) Patch helpers.py: skip Google-native files (.gsheet etc.) and tolerate single-file errors
hp = BASE / 'kit' / 'helpers.py'
src = hp.read_text(encoding='utf-8')
new_fn = '''def get_from_drive(src, dst):
    src = Path(src)
    dst = Path(dst)
    if src.suffix.lower() == ".zip":
        with zipfile.ZipFile(src) as z:
            z.extractall(dst.parent / "_unzip")
        root = next(iter(sorted((dst.parent / "_unzip").rglob("subjects.csv"))), None)
        if root is None:
            raise SystemExit("No subjects.csv inside the zip.")
        shutil.copytree(root.parent, dst, dirs_exist_ok=True)
        return
    skip_ext = {".gsheet", ".gdoc", ".gslides", ".gform", ".gdraw", ".gmap", ".gjam", ".gscript", ".tar.gz"}
    n_ok, skipped = 0, []
    for f in sorted(src.rglob("*")):
        rel = f.relative_to(src)
        try:
            if f.is_dir():
                (dst / rel).mkdir(parents=True, exist_ok=True)
                continue
        except OSError:
            continue
        if f.suffix.lower() in skip_ext or f.name.endswith(".tar.gz"):
            skipped.append(str(rel))
            continue
        out = dst / rel
        if out.exists() and out.stat().st_size == f.stat().st_size:
            n_ok += 1
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(f, out)
            n_ok += 1
        except OSError as e:
            skipped.append("%s (%s)" % (rel, e))
    print("Copied/kept %d files. Skipped %d: %s" % (n_ok, len(skipped), skipped[:8]))


'''
src = re.sub(r"def get_from_drive\(src, dst\):.*?(?=\ndef download_physionet)", new_fn.rstrip() + "\n\n", src, flags=re.S)
hp.write_text(src, encoding='utf-8')

# 2) Reload helpers and copy
import helpers as H
importlib.reload(H)
H.save_cfg(data='/content/cves-1.0.0', drive_data=DATASET_IN_DRIVE, download=bool(DOWNLOAD_FROM_PHYSIONET))
cfg = H.setup(need_data=False)
if not (cfg.data / 'subjects.csv').exists():
    if DATASET_IN_DRIVE:
        H.get_from_drive(DATASET_IN_DRIVE, cfg.data)
    elif DOWNLOAD_FROM_PHYSIONET:
        H.download_physionet(cfg.data)
    else:
        raise SystemExit('Set DATASET_IN_DRIVE in section 2, then re-run sections 2-4.')
H.check_data(cfg)

In [ ]:
#@title 5. Smoke test: known answers (about 5 seconds) { display-mode: "form" }
H.run_step(cfg, '0_known_answers', 'known_answers.py')
H.compare(cfg, '0_known_answers')
print('\nIf every line says PASS, go on to notebook 01. If a test fails, stop: do not patch around it.')

## Notes
- **Fixed folder.** The notebooks assume `MyDrive/CVES Fresh/CVES Repair/`. If you move it, edit `BASE` at the top of `helpers.py` in `kit/`.
- **Each new Colab session** starts with an empty `/content`. The first cell of notebooks 01–05 re-installs `wfdb` and restores the dataset from Drive (or PhysioNet).
- **Do not put the private folder in git.** The notebook checks this, as the kit's `run_all.sh` does.
- The kit and all analysis choices are the supervisor's (`ANALYSIS_LOCK.md` in `kit/`). These notebooks wrap them; they do not change the maths.